# Build a small Perturb-seq model-development subset

This notebook writes two small AnnData files that preserve the source formats and metadata:

- `DE_stats_model_subset.h5ad`: six QC-passing targets × three conditions, with 128 shared measured genes and the original DE layers and annotations.
- `gene_embedding_model_subset.h5ad`: embedding rows for those same 128 Ensembl gene IDs, with the original embedding dimensions and annotations.

The resulting matrices are small enough for early model prototyping on a MacBook Pro with 8 GB RAM. The source objects are opened in backed mode and only the selected rows and columns are copied into memory. The selection is deterministic. Change the paths in the next cell if the source files are stored elsewhere; `PERTURB_DATA_DIR` can also be set before launching Jupyter.


In [1]:
from pathlib import Path
import json
import os

import anndata as ad
import numpy as np
import pandas as pd

# Input files. Override PERTURB_DATA_DIR on another machine, or edit this path.
data_dir = Path(os.environ.get('PERTURB_DATA_DIR', '/work/magroup/skrieger/Syndri/perturb'))
de_path = data_dir / 'GWCD4i.DE_stats.h5ad'
embedding_path = data_dir / 'scGenePT_GO_BP_(D=1536).h5ad'

# Outputs are written under the current Jupyter working directory.
output_dir = Path.cwd() / 'model_architecture_subset'
output_dir.mkdir(parents=True, exist_ok=True)

target_limit = 6
feature_limit = 128
condition_order = ['Rest', 'Stim8hr', 'Stim48hr']
min_cells_per_condition = 75
preferred_min_guides = 2

for source_path in (de_path, embedding_path):
    if not source_path.exists():
        raise FileNotFoundError(
            f'{source_path} was not found. Set PERTURB_DATA_DIR to the folder containing both source files.'
        )


In [2]:
# Open the large source objects in backed mode; metadata stays small in memory.
de = ad.read_h5ad(de_path, backed='r')
embedding = ad.read_h5ad(embedding_path, backed='r')
required_obs = {
    'target_contrast', 'culture_condition', 'n_cells_target', 'n_guides',
    'ontarget_significant', 'distal_offtarget_flag', 'guide_correlation_all',
    'donor_correlation_all_mean',
}
missing = required_obs.difference(de.obs.columns)
if missing:
    raise ValueError(f'DE AnnData is missing required QC fields: {sorted(missing)}')
if 'gene_ids' not in de.var:
    raise KeyError("DE AnnData .var must contain Ensembl IDs in 'gene_ids'")
if 'ensembl_gene_id' not in embedding.obs:
    # In this source AnnData, the Ensembl ID may be the obs index rather than
    # a separate obs column. Use it only when the index is clearly Ensembl IDs.
    embedding_index = pd.Index(embedding.obs_names.astype(str))
    is_ensembl_index = embedding_index.str.match(r'^ENSG\d+(?:\.\d+)?$').all()
    if not is_ensembl_index:
        raise KeyError(
            "Could not find Ensembl IDs in embedding .obs['ensembl_gene_id'] or its observation index"
        )
    embedding.obs['ensembl_gene_id'] = embedding_index

# Normalize Ensembl IDs and source fields without changing the source objects.
obs = de.obs.copy()
obs['target_contrast'] = (
    obs['target_contrast'].astype('string').str.strip()
    .str.replace(r'\.\d+$', '', regex=True)
)
obs['culture_condition'] = obs['culture_condition'].astype('string').str.strip()
for column in [
    'n_cells_target', 'n_guides', 'guide_correlation_all', 'donor_correlation_all_mean'
]:
    obs[column] = pd.to_numeric(obs[column], errors='coerce')

def parse_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype('boolean')
    values = series.astype('string').str.strip().str.lower()
    return values.map({
        'true': True, '1': True, 'yes': True,
        'false': False, '0': False, 'no': False,
    }).astype('boolean')

obs['ontarget_significant'] = parse_bool(obs['ontarget_significant'])
obs['distal_offtarget_flag'] = parse_bool(obs['distal_offtarget_flag'])
obs = obs[obs['culture_condition'].isin(condition_order)].copy()

# Keep targets represented in all conditions, with cell coverage, at least one
# significant on-target result, and no distal off-target flag. Rank two-guide
# targets and positive correlations first as preferences.
summary = obs.groupby('target_contrast').agg(
    conditions_present=('culture_condition', 'nunique'),
    min_cells=('n_cells_target', 'min'),
    min_guides=('n_guides', 'min'),
    significant_conditions=('ontarget_significant', lambda x: int(x.fillna(False).sum())),
    any_distal_offtarget=('distal_offtarget_flag', lambda x: bool(x.isna().any() or x.fillna(True).any())),
    positive_guide_conditions=('guide_correlation_all', lambda x: int((x > 0).sum())),
    positive_donor_conditions=('donor_correlation_all_mean', lambda x: int((x > 0).sum())),
).reset_index()
summary['passes_core_qc'] = (
    summary['conditions_present'].eq(len(condition_order))
    & summary['min_cells'].ge(min_cells_per_condition)
    & summary['significant_conditions'].ge(1)
    & ~summary['any_distal_offtarget']
)
summary['preferred_guides'] = summary['min_guides'].ge(preferred_min_guides)

embedding_ids = (
    embedding.obs['ensembl_gene_id'].astype('string').str.strip()
    .str.replace(r'\.\d+$', '', regex=True)
)
embedding_id_set = set(embedding_ids.dropna())
summary['in_embedding'] = summary['target_contrast'].isin(embedding_id_set)
summary = summary[summary['passes_core_qc'] & summary['in_embedding']].copy()
summary = summary.sort_values(
    ['preferred_guides', 'significant_conditions', 'positive_donor_conditions',
     'positive_guide_conditions', 'target_contrast'],
    ascending=[False, False, False, False, True],
    na_position='last',
).reset_index(drop=True)

if len(summary) < target_limit:
    raise ValueError(f'Only {len(summary)} targets pass QC and map to the embedding; {target_limit} are required.')
selected_targets = summary['target_contrast'].head(target_limit).tolist()
selected_target_summary = summary.head(target_limit).copy()
print(f'Selected {len(selected_targets)} targets: {selected_targets}')
selected_target_summary


Selected 6 targets: ['ENSG00000004487', 'ENSG00000005339', 'ENSG00000005483', 'ENSG00000005812', 'ENSG00000008838', 'ENSG00000009307']


,target_contrast,conditions_present,min_cells,min_guides,significant_conditions,any_distal_offtarget,positive_guide_conditions,positive_donor_conditions,passes_core_qc,preferred_guides,in_embedding
0,ENSG00000004487,3,541.0,2.0,3,False,3,3,True,True,True
1,ENSG00000005339,3,1015.0,2.0,3,False,3,3,True,True,True
2,ENSG00000005483,3,222.0,2.0,3,False,3,3,True,True,True
3,ENSG00000005812,3,486.0,2.0,3,False,3,3,True,True,True
4,ENSG00000008838,3,669.0,2.0,3,False,3,3,True,True,True
5,ENSG00000009307,3,519.0,2.0,3,False,3,3,True,True,True


In [3]:
# Select up to 128 Ensembl IDs shared by the DE feature axis and embedding.
# Put selected target IDs first when they are measured features, then fill the
# remainder by sorted Ensembl ID for a stable, reproducible feature order.
de_var_ids = (
    de.var['gene_ids'].astype('string').str.strip()
    .str.replace(r'\.\d+$', '', regex=True)
)
de_var_ids = pd.Index(de_var_ids)
shared_ids = sorted(set(de_var_ids.dropna()) & embedding_id_set)
if len(shared_ids) < feature_limit:
    raise ValueError(f'Only {len(shared_ids)} shared Ensembl IDs are available; {feature_limit} are required.')

feature_ids = [gene_id for gene_id in selected_targets if gene_id in set(shared_ids)]
feature_ids += [gene_id for gene_id in shared_ids if gene_id not in set(feature_ids)]
feature_ids = feature_ids[:feature_limit]

# Map feature IDs and target-condition pairs to positions in the backed objects.
feature_to_de_position = {}
for position, gene_id in enumerate(de_var_ids):
    if pd.notna(gene_id) and gene_id not in feature_to_de_position:
        feature_to_de_position[gene_id] = position
de_var_positions = np.array([feature_to_de_position[gene_id] for gene_id in feature_ids])

obs_ids = obs['target_contrast'].astype('string')
obs_conditions = obs['culture_condition'].astype('string')
source_obs = de.obs
obs_positions = []
for target_id in selected_targets:
    for condition in condition_order:
        match = np.flatnonzero(
            (source_obs['target_contrast'].astype('string').to_numpy() == target_id)
            & (source_obs['culture_condition'].astype('string').to_numpy() == condition)
        )
        if len(match) != 1:
            raise ValueError(f'Expected one DE row for {target_id} in {condition}; found {len(match)}.')
        obs_positions.append(int(match[0]))

embedding_id_to_position = {}
for position, gene_id in enumerate(embedding_ids):
    if pd.notna(gene_id) and gene_id not in embedding_id_to_position:
        embedding_id_to_position[gene_id] = position
embedding_positions = np.array([embedding_id_to_position[gene_id] for gene_id in feature_ids])

print(f'DE subset shape: ({len(obs_positions)} perturbation-condition rows, {len(feature_ids)} genes)')
print(f'Embedding subset shape: ({len(embedding_positions)} genes, {embedding.n_vars} embedding dimensions)')


DE subset shape: (18 perturbation-condition rows, 128 genes)
Embedding subset shape: (128 genes, 1536 embedding dimensions)


In [4]:
# Materialize only the selected slices. The DE source stores X=None and its
# measurements in layers, so build that AnnData explicitly instead of calling
# to_memory() on a view (which tries to slice the null X matrix).
de_view = de[obs_positions, de_var_positions]
de_subset = ad.AnnData(
    X=None,
    obs=de_view.obs.copy(),
    var=de_view.var.copy(),
    uns=de.uns.copy(),
)
for layer_name in de.layers.keys():
    de_subset.layers[layer_name] = np.asarray(de_view.layers[layer_name]).copy()
for varm_name in de.varm.keys():
    de_subset.varm[varm_name] = de_view.varm[varm_name].copy()

# The embedding has a real expression matrix, so the selected backed view can
# be materialized directly.
embedding_subset = embedding[embedding_positions, :].to_memory()

# Store the shared feature ordering explicitly for downstream model code.
feature_order = np.arange(len(feature_ids), dtype=np.int32)
de_subset.var['model_feature_order'] = feature_order
de_subset.var['ensembl_gene_id'] = feature_ids
embedding_subset.obs['model_feature_order'] = feature_order
embedding_subset.obs['ensembl_gene_id'] = feature_ids

# The DE rows are stored target-major, then condition-major.
de_subset.obs['model_target_order'] = np.repeat(np.arange(len(selected_targets)), len(condition_order))
de_subset.obs['model_condition_order'] = condition_order * len(selected_targets)

# Save compact files; gzip keeps the tiny prototype bundle easy to move.
de_out = output_dir / 'DE_stats_model_subset.h5ad'
embedding_out = output_dir / 'gene_embedding_model_subset.h5ad'
de_subset.write_h5ad(de_out, compression='gzip')
embedding_subset.write_h5ad(embedding_out, compression='gzip')

manifest = {
    'source_de_file': de_path.name,
    'source_embedding_file': embedding_path.name,
    'de_subset_file': de_out.name,
    'embedding_subset_file': embedding_out.name,
    'target_ids': selected_targets,
    'conditions': condition_order,
    'feature_ids': feature_ids,
    'de_shape': list(de_subset.shape),
    'embedding_shape': list(embedding_subset.shape),
    'criteria': {
        'all_conditions_required': condition_order,
        'minimum_cells_per_condition': min_cells_per_condition,
        'significant_on_target_in_at_least_one_condition': True,
        'distal_offtarget_flag_must_be_false': True,
        'at_least_two_guides_and_positive_correlations_are_preferences': True,
    },
}
manifest_path = output_dir / 'subset_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n')

print(f'Saved {de_out} ({de_out.stat().st_size / 1e6:.2f} MB)')
print(f'Saved {embedding_out} ({embedding_out.stat().st_size / 1e6:.2f} MB)')
print(f'Saved {manifest_path}')


Saved /home/jwinchel/modsim-perturb/cmu-modeling-final-project/notebooks/model_architecture_subset/DE_stats_model_subset.h5ad (0.37 MB)
Saved /home/jwinchel/modsim-perturb/cmu-modeling-final-project/notebooks/model_architecture_subset/gene_embedding_model_subset.h5ad (1.15 MB)
Saved /home/jwinchel/modsim-perturb/cmu-modeling-final-project/notebooks/model_architecture_subset/subset_manifest.json


In [5]:
# Close the backed source files after writing the small prototype objects.
de.file.close()
embedding.file.close()

# Reopen the saved files to check their shapes and preserved fields.
de_check = ad.read_h5ad(de_out, backed='r')
embedding_check = ad.read_h5ad(embedding_out, backed='r')
print('DE:', de_check.shape, '| layers:', list(de_check.layers.keys()))
print('Embedding:', embedding_check.shape, '| obs fields:', list(embedding_check.obs.columns))
de_check.file.close()
embedding_check.file.close()


DE: (18, 128) | layers: ['adj_p_value', 'baseMean', 'lfcSE', 'log_fc', 'p_value', 'zscore']
Embedding: (128, 1536) | obs fields: ['symbol', 'ncbi_gene_id', 'biotype', 'description', 'synonyms', 'ensembl_gene_id', 'model_feature_order']
